# 02 - Silver Transformation

Applies explicit data types, trims and standardizes text, removes rows with missing business keys,
de-duplicates on the business key and rejects invalid sales rows.

In [ ]:
from pyspark.sql import functions as F

DATE_FMT = "yyyy-MM-dd"

customers = spark.table("bronze_customers")
products = spark.table("bronze_products")
sales = spark.table("bronze_sales")

silver_customers = (
    customers.select(
        F.trim("customer_id").alias("customer_id"),
        F.initcap(F.trim("customer_name")).alias("customer_name"),
        F.trim("segment").alias("segment"),
        F.initcap(F.trim("city")).alias("city"),
        F.initcap(F.trim("state")).alias("state"),
        F.trim("region").alias("region"),
        F.to_date(F.trim("join_date"), DATE_FMT).alias("join_date"),
    )
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id") != ""))
    .dropDuplicates(["customer_id"])
)

silver_products = (
    products.select(
        F.trim("product_id").alias("product_id"),
        F.trim("product_name").alias("product_name"),
        F.trim("category").alias("category"),
        F.col("unit_cost").cast("decimal(18,2)").alias("unit_cost"),
        F.col("list_price").cast("decimal(18,2)").alias("list_price"),
    )
    .filter(F.col("product_id").isNotNull() & (F.col("product_id") != ""))
    .filter(F.col("list_price") >= 0)
    .dropDuplicates(["product_id"])
)

silver_sales = (
    sales.select(
        F.trim("order_id").alias("order_id"),
        F.to_date(F.trim("order_date"), DATE_FMT).alias("order_date"),
        F.trim("customer_id").alias("customer_id"),
        F.trim("product_id").alias("product_id"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("unit_price").cast("decimal(18,2)").alias("unit_price"),
        F.col("discount_pct").cast("decimal(5,4)").alias("discount_pct"),  # stored as a fraction, e.g. 0.15 = 15%
        F.col("sales_amount").cast("decimal(18,2)").alias("sales_amount"),
        F.col("cost_amount").cast("decimal(18,2)").alias("cost_amount"),
    )
    .filter(F.col("order_id").isNotNull() & (F.col("order_id") != ""))
    .filter(
        F.col("order_date").isNotNull()
        & (F.col("quantity") > 0)
        & (F.col("sales_amount") >= 0)
        & F.col("cost_amount").isNotNull()
    )
    .dropDuplicates(["order_id"])
)

for name, df in {
    "silver_customers": silver_customers,
    "silver_products": silver_products,
    "silver_sales": silver_sales,
}.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)

# Row-count reconciliation: how many Bronze rows were rejected or de-duplicated
for bronze, silver in [("bronze_customers", "silver_customers"),
                       ("bronze_products", "silver_products"),
                       ("bronze_sales", "silver_sales")]:
    b, s = spark.table(bronze).count(), spark.table(silver).count()
    print(f"{silver}: {s:,} rows (bronze {b:,}, removed {b - s:,})")